# SCNN on TuSimple — Colab / A100

Trains SCNN lane detection end to end on a Colab GPU, driving the repo's own
`scnn_tusimple.py` rather than re-implementing the loop, so what runs here is the
same code path that runs on a workstation.

**Before you start:** Runtime → Change runtime type → **GPU / A100**.

Roughly 15–25 min on an A100 at the reference schedule (`batch 32 × 1500
iterations` = 48,000 images seen, matching `experiments/exp0`).

Order: check runtime → clone → data → smoke test → train → curves → score →
visualise → back up to Drive.

## 1. Runtime

In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU"
free, total = (v / 2**30 for v in torch.cuda.mem_get_info())
name = torch.cuda.get_device_name(0)
print(f"torch {torch.__version__}")
print(f"{name}: {total:.1f} GiB total, {free:.1f} GiB free")
if total - free > 0.5:
    print("warning: something already holds GPU memory; batch size will be scaled down")
if "A100" not in name:
    print(f"note: this is a {name}, not an A100 — the batch size adapts, timings will differ")

## 2. Repository and dependencies

Colab ships torch/torchvision/numpy/matplotlib. Only the four extras are needed:
`opencv-python`, `scikit-learn` (the TuSimple evaluator imports it), `tqdm`, `gdown`.

In [ ]:
REPO_URL = "https://github.com/harshwadhawe/SCNN_Culane.git"

import os
if not os.path.isdir("/content/SCNN_Culane"):
    !git clone -q $REPO_URL /content/SCNN_Culane
%cd /content/SCNN_Culane
!git pull -q && git log --oneline -1

In [ ]:
!pip install -q opencv-python scikit-learn tqdm gdown

In [ ]:
import torch, cv2, sklearn, numpy
print("cv2", cv2.__version__, "| sklearn", sklearn.__version__, "| numpy", numpy.__version__)
print("cuda:", torch.cuda.is_available())

## 3. Dataset

**Keep the data on local disk (`/content`), not on Drive.** Every epoch reads all
6,408 JPEGs; served over a mounted Drive that becomes the bottleneck and can turn
a 20-minute run into hours. Copy or download to `/content/TUSimple` first.

Pick one route below.

In [ ]:
DATA_ROOT = "/content/TUSimple"      # local disk, deliberately not /content/drive
SOURCE    = "kaggle"                 # "kaggle" | "drive" | "existing"

### 3a. Route: Kaggle

Needs an API token — kaggle.com → Settings → *Create New Token* → `kaggle.json`.

In [ ]:
if SOURCE == "kaggle":
    from google.colab import files
    print("upload kaggle.json")
    files.upload()
    !mkdir -p ~/.kaggle && mv -f kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
if SOURCE == "kaggle":
    !mkdir -p /content/dl
    !kaggle datasets download -d manideep1108/tusimple -p /content/dl
    !ls -la /content/dl

In [ ]:
if SOURCE == "kaggle":
    !mkdir -p $DATA_ROOT
    !for z in /content/dl/*.zip; do echo "unzip $z"; unzip -q -o "$z" -d $DATA_ROOT; done

### 3b. Route: Google Drive

Use this if you already uploaded the dataset (a zip, or an unpacked `TUSimple`
folder). Adjust `SRC` to match where it lives in your Drive.

In [ ]:
if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    SRC = "/content/drive/MyDrive/TUSimple"      # a folder, or a .zip

    !mkdir -p $DATA_ROOT
    if SRC.endswith(".zip"):
        !unzip -q -o "$SRC" -d $DATA_ROOT
    else:
        !cp -r "$SRC"/* $DATA_ROOT/

### 3c. Arrange and verify

Kaggle nests everything under `train_set/` and `test_set/`, but
`dataset/Tusimple.py` needs a single `clips/` at the root with the label files
beside it. `--arrange` flattens it, merging the date directories the two trees
share (`0531`, `0601`) at clip level rather than replacing them, then checks that
every image the labels reference actually exists.

In [ ]:
!python download_tusimple.py --dest $DATA_ROOT --inspect

In [ ]:
!python download_tusimple.py --dest $DATA_ROOT --arrange

You want all four lines `ok` with **0 images missing**, and these counts:

| file | entries |
|---|---|
| `label_data_0313.json` | 2858 |
| `label_data_0531.json` | 358 |
| `label_data_0601.json` | 410 |
| `test_label.json` | 2782 |

That is 3268 train / 358 val / 2782 test — 6408 labelled frames. Only frame
`20.jpg` of each clip is ever annotated, which is why this is far smaller than
the full TuSimple download.

## 4. Smoke test

Twenty iterations, no evaluation. Confirms CUDA, the dataloader, AMP and
checkpointing all work before committing to the real run.

**First run is slow to start:** `dataset/Tusimple.py` generates `seg_label/` for
all 6408 frames on first instantiation — a minute or two with no output. It is
cached afterwards.

In [ ]:
!python scnn_tusimple.py --data $DATA_ROOT --exp-dir /content/smoke --max-iter 20 --no-eval

## 5. Train

Defaults resolve from the hardware: on a 40 GB A100 that is **batch 32**, which
is exactly the `experiments/exp0` reference, so `lr` lands on 0.15 and
`max_iter 1500` sees the same 48,000 images as the published run.

`max_iter` counts **iterations, not images**. If the batch comes out smaller than
32 (a busy GPU, or a T4/V100), scale `max_iter` by `32/batch` to keep the same
number of images seen — otherwise you silently train on less data.

Colab drops long sessions. The script checkpoints every epoch and
`--resume` picks up from `latest.pth`, so re-running the cell after a disconnect
continues rather than restarting.

In [ ]:
EXP_DIR = "/content/exp_tusimple"
!python scnn_tusimple.py --data $DATA_ROOT --exp-dir $EXP_DIR --max-iter 1500

After a disconnect, run this instead of the cell above:

In [ ]:
# !python scnn_tusimple.py --data $DATA_ROOT --exp-dir $EXP_DIR --max-iter 1500 --resume

## 6. Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

hist = pd.read_csv(f"{EXP_DIR}/history.csv")
hist.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), layout="constrained")
axes[0].plot(hist.epoch, hist.train_loss, "o-", lw=1.4)
axes[0].set_xlabel("epoch"); axes[0].set_title("train loss")

axes[1].plot(hist.epoch, hist.val_loss, "o-", lw=1.4, color="tab:red")
axes[1].set_xlabel("epoch"); axes[1].set_title("val loss")

axes[2].plot(hist.epoch, hist.lr, lw=1.6, color="tab:green")
axes[2].set_xlabel("epoch"); axes[2].set_title("PolyLR")
for ax in axes:
    ax.grid(alpha=.25)
plt.show()
print(f"best val {hist.val_loss.min():.4f} at epoch {int(hist.val_loss.idxmin())} "
      f"| {hist.elapsed_s.iloc[-1]/60:.1f} min total")

## 7. Score

`LaneEval.bench_one_submit` runs automatically at the end of training. Re-run it
standalone with `--eval-only`, which loads `best.pth` rather than the last epoch.

Reference for `experiments/exp0`: **94.16% accuracy, FP 0.0735, FN 0.0825**.

In [ ]:
import json, pathlib

result = pathlib.Path(f"{EXP_DIR}/evaluation_result.txt")
if not result.exists():
    !python scnn_tusimple.py --data $DATA_ROOT --exp-dir $EXP_DIR --eval-only

In [ ]:
rows = json.loads(pathlib.Path(f"{EXP_DIR}/evaluation_result.txt").read_text())
ref = {"Accuracy": 0.9416, "FP": 0.0735, "FN": 0.0825}
print(f"{'metric':<10}{'this run':>12}{'exp0 ref':>12}")
for r in rows:
    print(f"{r['name']:<10}{r['value']:>12.4f}{ref.get(r['name'], float('nan')):>12.4f}")

## 8. Predictions on test images

In [ ]:
import numpy as np, torch, cv2
from model import SCNN
from utils.transforms import Compose, Resize, ToTensor, Normalize
from utils.prob2lines import getLane
import dataset

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
RESIZE = (512, 288)
device = torch.device("cuda")

net = SCNN(input_size=RESIZE, pretrained=False)
net.load_state_dict(torch.load(f"{EXP_DIR}/best.pth", map_location="cpu", weights_only=False)["net"])
net = net.to(device).eval()
print("loaded best.pth")

In [ ]:
# the repo's colour arrays are BGR (they end at cv2.imwrite); matplotlib draws RGB
LANE_COLORS = np.array([[255, 125, 0], [0, 255, 0], [0, 0, 255], [0, 255, 255]], np.uint8)[:, ::-1]

def overlay(rgb, mask, alpha=.8):
    lane = np.zeros_like(rgb)
    for i in range(4):
        lane[mask == i + 1] = LANE_COLORS[i]
    return cv2.addWeighted(lane, alpha, rgb, 1., 0.)

def predicted_mask(seg, exist, thresh=.5):
    mask = np.argmax(seg, axis=0)
    for i in range(4):
        if exist[i] <= thresh:
            mask[mask == i + 1] = 0
    return mask

In [ ]:
test_set = dataset.Tusimple(DATA_ROOT, "test",
                            Compose(Resize(RESIZE), ToTensor(), Normalize(MEAN, STD)))
picks = range(0, len(test_set), max(1, len(test_set) // 6))
batch = dataset.Tusimple.collate([test_set[i] for i in list(picks)[:6]])

with torch.no_grad():
    seg, exist = net(batch["img"].to(device))[:2]
    seg = torch.softmax(seg.float(), 1).cpu().numpy()
    exist = exist.float().cpu().numpy()
print("predicted on", len(seg), "test frames")

In [ ]:
import matplotlib.pyplot as plt

n = len(seg)
fig, axes = plt.subplots(n, 1, figsize=(9, 2.6 * n), layout="constrained")
for j, ax in enumerate(np.atleast_1d(axes)):
    rgb = cv2.cvtColor(cv2.imread(batch["img_name"][j]), cv2.COLOR_BGR2RGB)
    rgb = Resize(RESIZE)({"img": rgb})["img"]
    ax.imshow(overlay(rgb, predicted_mask(seg[j], exist[j])))
    ax.set_title(f"{'/'.join(batch['img_name'][j].split('/')[-3:])}   "
                 f"exist {(exist[j] > .5).astype(int).tolist()}", fontsize=8)
    ax.axis("off")
plt.show()

### Lane coordinates, as the evaluator sees them

In [ ]:
flags = [int(exist[0, k] > .5) for k in range(4)]
lanes = getLane.prob2lines_tusimple(seg[0], flags, resize_shape=(720, 1280), y_px_gap=10, pts=56)
print(f"{len(lanes)} lanes on {batch['img_name'][0].split('/')[-3:]}")
for i, l in enumerate(lanes):
    xs = [int(x) for x, _ in l if x > 0]
    print(f"  lane {i+1}: {len(xs)} points, x {xs[0] if xs else '-'} -> {xs[-1] if xs else '-'}")

## 9. Back up to Drive

`/content` is wiped when the runtime recycles. Copy the checkpoint out if you
want to keep it — `best.pth` is ~164 MB.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = "/content/drive/MyDrive/scnn_tusimple"
!mkdir -p $DEST
!cp $EXP_DIR/best.pth $EXP_DIR/history.csv $EXP_DIR/train.log $EXP_DIR/cfg.json $DEST/ 2>/dev/null
!cp $EXP_DIR/evaluation_result.txt $DEST/ 2>/dev/null
!ls -la $DEST

## Notes

- **Data on local disk, not Drive.** Re-reading 6408 JPEGs per epoch over a
  mounted Drive dominates runtime. `/content` is a local SSD.
- **`max_iter` counts iterations.** Change the batch size and scale it by
  `32/batch` or you train on proportionally less data.
- **Disconnects.** `--resume` continues from `latest.pth`; `seg_label/` is
  already cached so restarts are quick.
- **CULane next.** `download_culane.py --dest /content/CULane` fetches it, but it
  is ~55 GB and will not fit in a Colab session's disk — use a subset, or run it
  on the workstation.